# Run the translation server here, on this Colab GPU

This runs the translator's detection + OCR inside this notebook's VM
(a free T4 GPU needs only a Google sign-in — no card, no other
account) and opens a Cloudflare quick tunnel so the extension can
reach it.

Click **Runtime → Run all**, wait for the banner at the end, then paste
the two printed values into the extension (Options → Model → Where
detection runs → Cloud) and press **Test cloud & prewarm**.

Keep in mind:

- **Keep this tab open.** Colab reclaims the VM after ~90 minutes
  without tab activity, and a session ends by ~12 hours regardless.
- **The URL changes every session.** After a reconnect, run the cells
  again and paste the new URL into the extension.
- **Free T4s come from a shared quota** — not guaranteed. No GPU this
  time? Runtime → Change runtime type → T4 GPU, then Run all again;
  CPU still works, just much slower.
- Colab's free tier is for interactive use: fine while you read with
  the tab open, not for an unattended server.


## 0. Check the GPU


In [ ]:
import shutil, subprocess
if shutil.which("nvidia-smi"):
    r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print((r.stdout or r.stderr).strip()[:400])
    if r.returncode != 0 or "GPU" not in r.stdout:
        print("! GPU not visible to nvidia-smi — Runtime → Change runtime type → T4 GPU, then Run all again")
else:
    print("! no GPU runtime (nvidia-smi missing) — Runtime → Change runtime type → T4 GPU, then Run all again. Continuing on CPU.")


## 1. Install

Installs the inference stack here (a minute or two).


In [ ]:
%pip uninstall -q -y onnxruntime
# pinned: 1.30+ wheels need CUDA 13, Colab's T4 image ships CUDA 12.8
%pip install -q "onnxruntime-gpu==1.22.0" fastapi "uvicorn[standard]" pillow opencv-python-headless numpy
import onnxruntime
print("onnxruntime", onnxruntime.__version__, "| providers:", onnxruntime.get_available_providers())


## 2. Save the server files

Copies the server code here. Just run both cells.


In [ ]:
%%writefile app.py
# Cloud inference for the manga translator: CTD text detection + Baberu OCR
# over HTTP, CPU-only (fits the HuggingFace free tier).
#
# Recipes ported 1:1 from src/iframe/worker.ts — same thresholds, same gates,
# same decode loop. Resize uses bilinear like canvas drawImage; exact pixels
# may differ from the browser path, so parity is VERIFIED (not assumed) by
# comparing boxes/texts against the local pipeline on real pages.
#
# Panels: v1 returns [] — the client falls back to banding ordering, the same
# path it takes when the panel model file is missing. No fidelity risk.
# Splits: run_detect runs the same splitMergedBoxes family as the on-device
# worker (lane 1/2, twin-balloon cut, short-first, 10px split-input floor),
# and OCR crops grow past edge-cut glyphs like the client's expandCropToInk.
# /v1/page reports SPLIT_GEN so the client re-detects entries from older
# servers instead of rendering their fused boxes from cache.
import asyncio
import base64
import io
import json
import math
import os
import re
import time

import cv2
import numpy as np
import onnxruntime as ort
from fastapi import FastAPI, Query, Request
from fastapi.responses import JSONResponse
from PIL import Image

MODEL_DIR = os.environ.get("MODEL_DIR", "/models")
# EP chain: local default CPU; Modal sets "CUDAExecutionProvider,CPUExecutionProvider"
ORT_PROVIDERS = os.environ.get("ORT_PROVIDERS", "CPUExecutionProvider").split(",")

# ---- tunables: mirror src/iframe/worker.ts exactly ----
CTD_INPUT = 1024
CONF_THR = 0.35
NMS_THR = 0.35
MASK_THR = 0.3
MIN_SIZE = 12
LETTERBOX = (113, 113, 113)  # #717171
STRIP_ASPECT = 3
TILE_SIZE = 1200
TILE_OVERLAP = 180
COMP_GAP = 28

BABERU_MEAN = (0.485, 0.456, 0.406)
BABERU_STD = (0.229, 0.224, 0.225)
PAST_IN = [f"past_k{i}" for i in range(6)] + [f"past_v{i}" for i in range(6)]
PRESENT_OUT = [f"present_k{i}" for i in range(6)] + [f"present_v{i}" for i in range(6)]

app = FastAPI(title="arn-manga")
lock = asyncio.Lock()  # one inference at a time (2 vCPU, no oversubscription)
ctd = None
baberu = None  # {vis, pre, stp, bos, eos, id2ch, contentIds}
inpaint = None  # manga-LaMa fp16w; missing file only disables POST /v1/inpaint
EPS = {}  # session -> provider chain (proves GPU placement in prod logs)


def _load(path):
    if not os.path.isfile(path):
        raise RuntimeError(f"model file missing: {path}")
    return path


OPTS = ort.SessionOptions()


def _sess(path, providers):
    t = time.perf_counter()
    s = ort.InferenceSession(_load(path), OPTS, providers=providers)
    print(f"session {os.path.basename(path)}: {(time.perf_counter()-t)*1000:.0f}ms",
          flush=True)
    return s


def load_models():
    global ctd, baberu, inpaint
    print("onnxruntime:", ort.__version__,
          "available:", ort.get_available_providers(), flush=True)
    t_all = time.perf_counter()
    ctd = _sess(f"{MODEL_DIR}/ctd.onnx", ORT_PROVIDERS)
    vis = _sess(f"{MODEL_DIR}/vision-int4.onnx", ORT_PROVIDERS)
    pre = _sess(f"{MODEL_DIR}/baberu-prefill.onnx", ORT_PROVIDERS)
    stp = _sess(f"{MODEL_DIR}/baberu-step.onnx", ORT_PROVIDERS)
    print(f"all sessions: {(time.perf_counter()-t_all)*1000:.0f}ms", flush=True)
    with open(_load(f"{MODEL_DIR}/vocab.json"), encoding="utf-8") as f:
        charset = json.load(f)
    id2ch, content = {}, set()
    for i, ch in enumerate(charset):
        id2ch[i + 4] = ch
        # pass 1 like baberuParseVocab: single alnum, minus the long-dash set
        if len(ch) == 1 and ch not in "ーｰ〜~" and re.match(r"[A-Za-z0-9]", ch):
            content.add(i + 4)
    # pass 2 like the isContentChar extension (reference unicodedata approx)
    for i, ch in id2ch.items():
        cp = ord(ch[0]) if ch else 0
        if (re.match(r"[A-Za-z0-9]", ch) or 0x3040 <= cp <= 0x30FF
                or 0x3400 <= cp <= 0x9FFF or 0xF900 <= cp <= 0xFAFF
                or 0xFF66 <= cp <= 0xFF9D):
            content.add(i)
    baberu = {"vis": vis, "pre": pre, "stp": stp, "bos": 1, "eos": 2,
              "id2ch": id2ch, "contentIds": content}
    # optionally loadable: a server without the 112MB cleanup model still
    # serves detection/OCR, /v1/inpaint answers 503
    try:
        inpaint = _sess(f"{MODEL_DIR}/lama-manga-512-fp16w.onnx", ORT_PROVIDERS)
    except Exception as e:
        inpaint = None
        print(f"inpaint model not loaded: {e}", flush=True)
    EPS.update({n: s.get_providers() for n, s in
                {"ctd": ctd, "vis": vis, "pre": pre, "stp": stp}.items()
                if n != "inpaint"})
    if inpaint is not None:
        EPS["inpaint"] = inpaint.get_providers()
    print("ORT providers:", EPS, flush=True)


@app.on_event("startup")
def _startup():
    load_models()


@app.get("/health")
def health():
    # splitGen is here (not just /v1/page) so a remote client can verify the
    # server runs the split/mask logic its cache gate expects — a stale Colab
    # process otherwise fails silently back to fused boxes.
    return {"ok": bool(ctd and baberu), "device": os.environ.get("ORT_DEVICE", "cpu"),
            "panels": "client-fallback", "ep": EPS or None, "splitGen": SPLIT_GEN}


@app.get("/")
def root():
    return {"service": "arn-manga",
            "endpoints": ["/health", "POST /v1/page", "POST /v1/inpaint"]}


def nms(boxes, confs):
    idx = sorted(range(len(boxes)), key=lambda i: -confs[i])
    keep = []
    while idx:
        i = idx.pop(0)
        keep.append(i)
        a = boxes[i]
        rest = []
        for j in idx:
            b = boxes[j]
            ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
            iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
            inter = ix * iy
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if not (union > 0 and inter / union > NMS_THR):
                rest.append(j)
        idx = rest
    return keep


def split_tiles(w, h):
    vertical = h >= w
    long, short = (h, w) if vertical else (w, h)
    if long / short <= STRIP_ASPECT:
        return []
    step = TILE_SIZE - TILE_OVERLAP
    n = max(2, -(-(long - TILE_OVERLAP) // step))  # ceil
    ln = (long + (n - 1) * TILE_OVERLAP) / n
    out = []
    for i in range(n):
        o = round(i * (ln - TILE_OVERLAP))
        out.append((0, o, w, round(ln)) if vertical else (o, 0, round(ln), h))
    return out


def merge_tile_boxes(tiled):
    allb = [(b[0] + t[0], b[1] + t[1], b[2] + t[0], b[3] + t[1], b[4], ti)
            for ti, (t, bs) in enumerate(tiled) for b in bs]
    changed = True
    while changed:
        changed = False
        for i in range(len(allb)):
            for j in range(i + 1, len(allb)):
                a, b = allb[i], allb[j]
                if a[5] == b[5]:
                    continue
                gx = max(0, min(a[2], b[2]) - max(a[0], b[0]))
                gy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
                gapx = max(a[0] - b[2], b[0] - a[2], 0)
                gapy = max(a[1] - b[3], b[1] - a[3], 0)
                if not ((gx > 0 or gapx <= 8) and (gy > 0 or gapy <= 8)):
                    continue
                minside = min(a[2] - a[0], a[3] - a[1], b[2] - b[0], b[3] - b[1])
                if max(gx, gy) < 0.5 * minside:
                    continue
                allb[i] = (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]),
                           max(a[3], b[3]), max(a[4], b[4]), a[5])
                del allb[j]
                changed = True
                break
            if changed:
                break
    return [(x1, y1, x2, y2, c) for x1, y1, x2, y2, c, _ in allb]


from split import (SPLIT_GEN, expand_crop_to_ink, pack_mask, rescue_split_comp,
                   split_merged_boxes)
def infer_once(pil, conf_thr):
    w, h = pil.size
    s = CTD_INPUT / max(w, h)
    nw, nh = round(w * s), round(h * s)
    canvas = Image.new("RGB", (CTD_INPUT, CTD_INPUT), LETTERBOX)
    canvas.paste(pil.resize((nw, nh), Image.BILINEAR), (0, 0))
    x = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    t0 = time.perf_counter()
    names = [o.name for o in ctd.get_outputs()]
    out = dict(zip(names, ctd.run(None, {"image": x})))
    infer_ms = (time.perf_counter() - t0) * 1000
    raw = out["bbox_preds"].reshape(-1, 7)
    boxes, confs, low_boxes, low_confs = [], [], [], []
    for cx, cy, bw, bh, c4, c5, c6 in raw:
        conf = float(c4 * max(c5, c6))
        if conf < 0.05:
            continue
        bx = [(cx - bw / 2) / s, (cy - bh / 2) / s,
              (cx + bw / 2) / s, (cy + bh / 2) / s]
        low_boxes.append(bx)
        low_confs.append(conf)
        if conf < conf_thr:
            continue
        boxes.append(bx)
        confs.append(conf)
    # mask: top-left nw×nh of the 1024 field, upscaled to page size
    m = out["mask"].reshape(CTD_INPUT, CTD_INPUT)[:nh, :nw]
    prob = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    return boxes, confs, low_boxes, low_confs, prob, infer_ms


def run_detect(pil, conf_thr, min_size):
    w, h = pil.size
    tiles = split_tiles(w, h)
    infer_ms = 0.0
    if not tiles:
        boxes, confs, low_boxes, low_confs, prob, infer_ms = infer_once(pil, conf_thr)
    else:
        per = []
        prob = np.zeros((h, w), np.float32)
        for (x0, y0, tw, th) in tiles:
            b, c, lb, lc, p, ms = infer_once(
                pil.crop((x0, y0, x0 + tw, y0 + th)), conf_thr)
            per.append(((x0, y0), b, c, lb, lc, p))
            infer_ms += ms
        merged = merge_tile_boxes(
            [((x0, y0), [(*bb[:4], cc) for bb, cc in zip(b, c)])
             for (x0, y0), b, c, _, _, _ in per])
        boxes = [[x1, y1, x2, y2] for x1, y1, x2, y2, _ in merged]
        confs = [c for _, _, _, _, c in merged]
        low_boxes, low_confs = [], []
        for (x0, y0), _, _, lb, lc, _ in per:
            for l, c in zip(lb, lc):
                low_boxes.append([l[0] + x0, l[1] + y0, l[2] + x0, l[3] + y0])
                low_confs.append(c)
        for (x0, y0), _, _, _, _, p in per:
            th, tw = p.shape
            np.maximum(prob[y0:y0 + th, x0:x0 + tw], p,
                       out=prob[y0:y0 + th, x0:x0 + tw])
    keep = [i for i in nms(boxes, confs)
            if boxes[i][2] - boxes[i][0] > min_size and boxes[i][3] - boxes[i][1] > min_size]
    # containment gate: ≥80% inside another → drop the lower-confidence one
    contained = set()
    for a in range(len(keep)):
        for b in range(len(keep)):
            if a == b or a in contained or b in contained:
                continue
            A, B = boxes[keep[a]], boxes[keep[b]]
            ix = max(0, min(A[2], B[2]) - max(A[0], B[0]))
            iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
            inter = ix * iy
            if not inter:
                continue
            aA = (A[2] - A[0]) * (A[3] - A[1])
            aB = (B[2] - B[0]) * (B[3] - B[1])
            if inter > 0.8 * aA:
                contained.add(a if confs[keep[a]] <= confs[keep[b]] else b)
            elif inter > 0.8 * aB:
                contained.add(b if confs[keep[b]] < confs[keep[a]] else a)
    out_boxes = [
        {"x1": max(0.0, boxes[i][0]), "y1": max(0.0, boxes[i][1]),
         "x2": min(float(w), boxes[i][2]), "y2": min(float(h), boxes[i][3]),
         "conf": confs[i]}
        for i in keep if i not in contained
    ]

    def overlaps(c):
        for o in out_boxes:
            ix = max(0, min(o["x2"], c[2]) - max(o["x1"], c[0]))
            iy = max(0, min(o["y2"], c[3]) - max(o["y1"], c[1]))
            inter = ix * iy
            if inter > 0.05 * (c[2] - c[0]) * (c[3] - c[1]) or \
               inter > 0.15 * (o["x2"] - o["x1"]) * (o["y2"] - o["y1"]):
                return True
        return False

    # Same text-likelihood definition everywhere a mask component must claim
    # to be text: mean raw mask prob, corroborated by any low-confidence
    # box-head prediction overlapping it.
    def comp_box_conf(c):
        box_conf = 0.0
        c_area = (c["x2"] - c["x1"]) * (c["y2"] - c["y1"])
        for bx, bc in zip(low_boxes, low_confs):
            ix = max(0, min(bx[2], c["x2"]) - max(bx[0], c["x1"]))
            iy = max(0, min(bx[3], c["y2"]) - max(bx[1], c["y1"]))
            if ix * iy > 0.1 * c_area and bc > box_conf:
                box_conf = bc
        return box_conf

    # mask components (4-connectivity like the browser BFS)
    packed = (prob > MASK_THR).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(packed, connectivity=4)
    prob_sum = np.bincount(labels.ravel(), weights=prob.ravel(), minlength=n)
    comps = []
    for lab in range(1, min(n, 401)):
        x, y, bw, bh, area = (int(stats[lab, i]) for i in range(5))
        if bw >= 8 and bh >= 8:
            comps.append({"x1": x, "y1": y, "x2": x + bw, "y2": y + bh,
                          "count": int(area), "psum": float(prob_sum[lab]),
                          "labs": {lab}})
    # Split-input comps: raw text clusters snapshotted BEFORE the merge below
    # (a merged bbox would hide the gap between two balloons) and filtered by
    # the same text-likelihood gate pass 3 uses — mirrors worker.ts, including
    # the 10px split-evidence floor (pass 3 keeps its own >=14 floor, so no
    # new junk regions are created by this). box_comps is the stricter set the
    # child BOXES are measured from (mean prob >= 0.75).
    texty_comps, box_comps = [], []
    for c in comps:
        bw, bh = c["x2"] - c["x1"], c["y2"] - c["y1"]
        if bw < 10 or bh < 10:
            continue
        if c["count"] / (bw * bh) < 0.02:
            continue
        mean = c["psum"] / c["count"]
        if mean < 0.75 and comp_box_conf(c) < 0.20:
            continue
        r = {"x1": c["x1"], "y1": c["y1"], "x2": c["x2"], "y2": c["y2"]}
        texty_comps.append(r)
        if mean >= 0.75:
            box_comps.append(r)
    # merge touching-when-padded components
    changed = True
    while changed:
        changed = False
        for i in range(len(comps)):
            for j in range(i + 1, len(comps)):
                a, b = comps[i], comps[j]
                if not (a["x1"] - COMP_GAP > b["x2"] or b["x1"] - COMP_GAP > a["x2"]
                        or a["y1"] - COMP_GAP > b["y2"] or b["y1"] - COMP_GAP > a["y2"]):
                    comps[i] = {"x1": min(a["x1"], b["x1"]), "y1": min(a["y1"], b["y1"]),
                                "x2": max(a["x2"], b["x2"]), "y2": max(a["y2"], b["y2"]),
                                "count": a["count"] + b["count"], "psum": a["psum"] + b["psum"],
                                "labs": a["labs"] | b["labs"]}
                    del comps[j]
                    changed = True
                    break
            if changed:
                break
    page_area = w * h
    mask_boxes = []

    def overlaps_rect(r):
        for o in out_boxes:
            ix = max(0, min(o["x2"], r["x2"]) - max(o["x1"], r["x1"]))
            iy = max(0, min(o["y2"], r["y2"]) - max(o["y1"], r["y1"]))
            inter = ix * iy
            if inter > 0.05 * (r["x2"] - r["x1"]) * (r["y2"] - r["y1"]) or \
               inter > 0.15 * (o["x2"] - o["x1"]) * (o["y2"] - o["y1"]):
                return True
        return False

    for c in comps:
        if len(mask_boxes) >= 16:
            break
        x1, y1, x2, y2 = c["x1"], c["y1"], c["x2"], c["y2"]
        bw, bh = x2 - x1, y2 - y1
        fill = c["count"] / (bw * bh)
        if bw < 14 or bh < 14 or fill < 0.02 or fill > 0.6:
            continue
        if bw * bh > 0.2 * page_area:
            continue
        mask_prob = c["psum"] / c["count"]
        if mask_prob < 0.75 and comp_box_conf(c) < 0.20:
            continue
        if not overlaps((x1, y1, x2, y2)):
            mask_boxes.append({"x1": float(x1), "y1": float(y1),
                               "x2": float(x2), "y2": float(y2), "conf": 0.5})
            continue
        # sole killer was the overlap gate — second chance via split: pieces
        # outside all boxes survive as their own regions (see rescue_split_comp)
        labs = c["labs"]

        def count_in(rx1, ry1, rx2, ry2, _labs=labs):
            win_lab = labels[max(0, ry1):min(h, ry2), max(0, rx1):min(w, rx2)]
            win_pr = prob[max(0, ry1):min(h, ry2), max(0, rx1):min(w, rx2)]
            m = np.isin(win_lab, list(_labs))
            return int(m.sum()), float(win_pr[m].sum())

        for r in rescue_split_comp(
                {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                texty_comps, box_comps, COMP_GAP, page_area,
                count_in, overlaps_rect, comp_box_conf):
            if len(mask_boxes) >= 16:
                break
            mask_boxes.append({"x1": float(r["x1"]), "y1": float(r["y1"]),
                               "x2": float(r["x2"]), "y2": float(r["y2"]),
                               "conf": 0.5})
    # split AFTER the mask-only pass: a merged box's generous coverage must
    # still suppress mask clusters it swallowed (pre-split list feeds the
    # overlap gate), and only then does each balloon become its own box —
    # mirrors worker.ts.
    boxes = split_merged_boxes(out_boxes + mask_boxes, texty_comps, COMP_GAP, box_comps)
    # packed is 0/1 — packMask mirrors the client's byte mask (0/255)
    mw, mh, mbytes = pack_mask(w, h, (packed * 255).ravel())
    return boxes, infer_ms, {"w": mw, "h": mh,
                             "b64": base64.b64encode(mbytes).decode("ascii")}


def run_baberu(crop):
    B = baberu
    x = np.asarray(crop.resize((224, 224), Image.BICUBIC),
                   dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    mean = np.array(BABERU_MEAN, np.float32).reshape(1, 3, 1, 1)
    std = np.array(BABERU_STD, np.float32).reshape(1, 3, 1, 1)
    t0 = time.perf_counter()
    (embeds,) = B["vis"].run(["vision_embeds"], {"pixel_values": x})
    if not all(np.isfinite(embeds.flat[:16])):
        raise RuntimeError("baberu vision produced non-finite embeds")
    out = B["pre"].run(None, {
        "vision_embeds": embeds,
        "input_ids": np.array([[B["bos"]]], dtype=np.int64)})
    names = [o.name for o in B["pre"].get_outputs()]

    def last_logits(vals):
        lg, shape = vals[names.index("logits")], B["pre"].get_outputs()[names.index("logits")].shape
        return lg.reshape(-1, shape[-1])[-1].astype(np.float64)

    logits = last_logits(out)
    present = [out[names.index(n)] for n in PRESENT_OUT]
    seq, toks = [B["bos"]], []
    pos = embeds.shape[1] + 1
    for _ in range(128):
        for tid in set(seq):
            s = logits[tid]
            logits[tid] = s * 1.2 if s < 0 else s / 1.2
        if toks and toks[-1] in B["contentIds"]:
            last, run = toks[-1], 0
            for t in reversed(toks):
                if t != last:
                    break
                run += 1
            if run >= 12:
                logits[last] = -np.inf
        nxt = int(np.argmax(logits[1:]) + 1)
        if nxt == B["eos"]:
            break
        toks.append(nxt)
        seq.append(nxt)
        if len(toks) >= 128:
            break
        feed = {"input_ids": np.array([[nxt]], dtype=np.int64),
                "position_ids": np.array([[pos]], dtype=np.int64)}
        for nm, p in zip(PAST_IN, present):
            feed[nm] = p
        out = B["stp"].run(None, feed)
        snames = [o.name for o in B["stp"].get_outputs()]
        lg = out[snames.index("logits")]
        v = B["stp"].get_outputs()[snames.index("logits")].shape[-1]
        logits = lg.reshape(-1, v)[-1].astype(np.float64)
        present = [out[snames.index(n)] for n in PRESENT_OUT]
        pos += 1
    ms = (time.perf_counter() - t0) * 1000
    return "".join(B["id2ch"].get(t, "") for t in toks), ms


def baberu_crop(pil, rgb, b):
    pad = max(4, (b["y2"] - b["y1"]) * 0.10)
    rect = {"x": max(0, math.floor(b["x1"] - pad)), "y": max(0, math.floor(b["y1"] - pad)),
            "w": math.ceil(b["x2"] - b["x1"] + 2 * pad), "h": math.ceil(b["y2"] - b["y1"] + 2 * pad)}
    r = expand_crop_to_ink(rgb, b, rect)
    x = max(0, math.floor(r["x"]))
    y = max(0, math.floor(r["y"]))
    w = min(pil.width - x, math.ceil(r["w"]))
    h = min(pil.height - y, math.ceil(r["h"]))
    return pil.crop((x, y, x + w, y + h))


INPAINT_SIZE = 512
INPAINT_PAD_RATIO = 0.5


def inpaint_dilate_radius(w, h):
    """Mirror of aiCleanupDilate() in src/content/inpaint.ts — big scans have
    bigger glyph gaps, and a tight mask makes the model paint the leftover white
    glyphs over the whole window."""
    return min(10, max(4, round(4 * max(w, h) / 1600)))


def run_inpaint(pil, boxes, pad_ratio, mask=None):
    """Erase the given boxes with the manga-LaMa model (fp16 weights, 512x512).

    The client sends the prepared binary erase mask (restricted to the erase
    boxes and dilated — thin glyph strokes and the gaps between them drop out of
    the 512px window resize and the model then paints the leftover white glyphs'
    background over the whole window). Without one the mask is rebuilt from CTD
    here, restricted to the boxes and dilated with the same recipe. Windows are
    cut from the original image, edge-padded to a square, run at 512x512, and
    composited back only where the mask says text was. Returns per-box PNG
    patches (the same shape the on-device worker produces).
    """
    det_ms = 0.0
    if mask is None:
        _, _, _, _, prob, det_ms = infer_once(pil, CONF_THR)
        raw = prob > MASK_THR
        mask = np.zeros_like(raw)
        for b in boxes:
            x1 = max(0, int(np.floor(b["x1"]))); y1 = max(0, int(np.floor(b["y1"])))
            x2 = min(pil.width, int(np.ceil(b["x2"]))); y2 = min(pil.height, int(np.ceil(b["y2"])))
            mask[y1:y2, x1:x2] = raw[y1:y2, x1:x2]
        mask = cv2.dilate(mask.astype(np.uint8), np.ones((3, 3), np.uint8),
                          iterations=inpaint_dilate_radius(pil.width, pil.height)) > 0
    rgb = np.asarray(pil.convert("RGB"), dtype=np.uint8)
    H, W = rgb.shape[:2]
    out = rgb.copy()
    t0 = time.perf_counter()
    windows = 0
    for b in boxes:
        x1 = max(0, min(W - 1, int(np.floor(b["x1"]))))
        y1 = max(0, min(H - 1, int(np.floor(b["y1"]))))
        x2 = max(x1 + 1, min(W, int(np.ceil(b["x2"]))))
        y2 = max(y1 + 1, min(H, int(np.ceil(b["y2"]))))
        bw, bh = x2 - x1, y2 - y1
        pad = max(8, round(max(bw, bh) * pad_ratio))
        side = round(max(bw, bh) + 2 * pad)
        sx = round(x1 + bw / 2 - side / 2)
        sy = round(y1 + bh / 2 - side / 2)
        cx1, cy1 = max(0, sx), max(0, sy)
        cx2, cy2 = min(W, sx + side), min(H, sy + side)
        top, left = cy1 - sy, cx1 - sx
        bottom = side - (cy2 - cy1) - top
        right = side - (cx2 - cx1) - left
        crop = rgb[cy1:cy2, cx1:cx2]
        mcrop = mask[cy1:cy2, cx1:cx2].astype(np.uint8) * 255
        # edge padding (not reflect): always valid however wide the margin is
        crop_sq = np.pad(crop, ((top, bottom), (left, right), (0, 0)), mode="edge")
        mask_sq = np.pad(mcrop, ((top, bottom), (left, right)), mode="edge")
        img512 = np.asarray(Image.fromarray(crop_sq).resize(
            (INPAINT_SIZE, INPAINT_SIZE), Image.LANCZOS), dtype=np.float32) / 255.0
        m512 = np.asarray(Image.fromarray(mask_sq).resize(
            (INPAINT_SIZE, INPAINT_SIZE), Image.NEAREST)) > 127
        inp = np.concatenate([img512 * (1 - m512[..., None]),
                              m512[..., None].astype(np.float32)], axis=2)
        pred = inpaint.run(None, {"input": np.transpose(inp, (2, 0, 1))[None].astype(np.float32)})[0][0]
        pred = np.transpose(np.clip(pred, 0.0, 1.0), (1, 2, 0))
        win = np.asarray(Image.fromarray((pred * 255).astype(np.uint8)).resize(
            (side, side), Image.LANCZOS))
        mwin = np.asarray(Image.fromarray(mask_sq).resize(
            (side, side), Image.NEAREST)) > 127
        ox, oy = max(0, -sx), max(0, -sy)
        sub_out = out[cy1:cy2, cx1:cx2]
        sub_win = win[oy:oy + (cy2 - cy1), ox:ox + (cx2 - cx1)]
        sub_mask = mwin[oy:oy + (cy2 - cy1), ox:ox + (cx2 - cx1)]
        sub_out[sub_mask] = sub_win[sub_mask]
        windows += 1
    patches = []
    for b in boxes:
        px1 = max(0, int(np.floor(b["x1"])) - 4)
        py1 = max(0, int(np.floor(b["y1"])) - 4)
        px2 = min(W, int(np.ceil(b["x2"])) + 4)
        py2 = min(H, int(np.ceil(b["y2"])) + 4)
        crop = out[py1:py2, px1:px2]
        if crop.size == 0:
            continue
        buf = io.BytesIO()
        Image.fromarray(crop).save(buf, format="PNG")
        patches.append({"x1": px1, "y1": py1, "x2": px2, "y2": py2,
                        "png": base64.b64encode(buf.getvalue()).decode("ascii")})
    ms = (time.perf_counter() - t0) * 1000
    return patches, windows, ms, det_ms


@app.post("/v1/inpaint")
async def inpaint_page(req: Request, pad_ratio: float = Query(INPAINT_PAD_RATIO)):
    if inpaint is None:
        return JSONResponse({"ok": False, "error": "inpaint model not loaded on the server"}, 503)
    t0 = time.perf_counter()
    mask = None
    try:
        body = await req.json()
        raw = base64.b64decode(body.get("image") or "")
        boxes = body.get("boxes") or []
        pil = Image.open(io.BytesIO(raw)).convert("RGB")
        mask_b64 = body.get("mask")
        if mask_b64:
            m = Image.open(io.BytesIO(base64.b64decode(mask_b64))).convert("L")
            if m.size != pil.size:
                return JSONResponse({"ok": False, "error": f"mask size {m.size} != image {pil.size}"}, 400)
            mask = np.asarray(m) > 127
    except Exception as e:
        return JSONResponse({"ok": False, "error": f"bad request: {e}"}, 400)
    async with lock:
        try:
            patches, windows, ms, det_ms = run_inpaint(pil, boxes, pad_ratio, mask)
        except Exception as e:
            return JSONResponse({"ok": False, "error": f"inpaint failed: {e}"}, 500)
    return {"ok": True, "patches": patches, "windows": windows,
            "ms": {"detect": round(det_ms, 1), "inpaint": round(ms, 1),
                   "total": round((time.perf_counter() - t0) * 1000, 1)}}


@app.post("/v1/page")
async def page(req: Request,
               conf_thr: float = Query(CONF_THR), min_size: int = Query(MIN_SIZE)):
    t0 = time.perf_counter()
    raw = await req.body()
    body_ms = (time.perf_counter() - t0) * 1000
    try:
        pil = Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception as e:
        return JSONResponse({"ok": False, "error": f"bad image: {e} (got {len(raw)} bytes head={raw[:8].hex()})"}, 400)
    async with lock:
        boxes, det_ms, mask = run_detect(pil, conf_thr, min_size)
        rgb = np.asarray(pil.convert("RGB"), dtype=np.uint8)
        texts, ocr_ms = [], 0.0
        for b in boxes:
            try:
                t, ms = run_baberu(baberu_crop(pil, rgb, b))
            except Exception:
                t, ms = "", 0.0
            texts.append(t)
            ocr_ms += ms
    total = (time.perf_counter() - t0) * 1000
    return {
        "ok": True,
        "w": pil.width, "h": pil.height,
        "boxes": [{"x1": round(float(b["x1"]), 1), "y1": round(float(b["y1"]), 1),
                   "x2": round(float(b["x2"]), 1), "y2": round(float(b["y2"]), 1),
                   "conf": round(float(b["conf"]), 4)} for b in boxes],
        "panels": [],
        "panelSkipped": "cloud-v1: panel runs client-side, banding fallback applies",
        "splitGen": SPLIT_GEN,
        "mask": mask,
        "texts": texts,
        "ms": {"body": round(body_ms, 1), "detect": round(det_ms, 1),
               "ocr": round(ocr_ms, 1), "total": round(total, 1)},
    }


In [ ]:
%%writefile split.py
# Pure box-splitting + OCR-crop expansion for the cloud server — ported
# 1:1 from src/content/detection.ts (splitMergedBoxes family) and
# src/content/render.ts (expandCropToInk). Standard library only, so
# unit tests import this without the server's third-party deps; app.py
# is the only runtime importer (Modal/Colab/Docker all colocate it).
import math


# ---- box splitting (ported 1:1 from src/content/detection.ts) ----
# The box head fuses stacked/kissing balloons into one box; the mask
# components are the evidence that splits them back apart. Same lanes, same
# constants, same decisions. Two JS-isms to preserve:
#   - Math.round rounds half UP on positives; Python round() is banker's, so
#     _r() is used everywhere the TS calls Math.round (inputs here are >= 0).
#   - medians are the UPPER middle (sorted[floor(n/2)]), same in _med().
# Boxes are dicts (x1/y1/x2/y2/conf, +clip/cutAxis on split children);
# comps are plain rect dicts (the SplitComp shape — no count/psum).
SPLIT_GEN = 4  # bump when this section's logic changes; /v1/page reports it
# and the client re-detects cache entries written by older servers.
# gen 2: OCR crops grow past edge-cut glyphs + /v1/page ships the packed CTD
# mask (gen 1 split without it — the client's box-filled stand-in mask forced
# white text on every leaked area).
# gen 3: pass-3 rescue — a merged comp killed only by the overlap gate is
# split and re-gated per piece (live /14: the merge chained the left はむ
# into a super-comp that box 5 swallowed whole).
# gen 4: lane-2 first-pair — a detached FIRST group of comparable size splits
# despite nesting (live /14 right group: 3-row hamu 34px above its EN block).
# Stragglers (small group under a big block, the dropped-line family) stay
# fused via the size ratio.
SPLIT_GAP_FACTOR = 2
SPLIT_GAP_RATIO = 0.8
SPLIT_PAD_CAP = 40
SPLIT2_FLOOR_RATIO = 0.5
SPLIT2_FLOOR_MIN = 8
SPLIT2_OVERLAP_MAX = 0.5
SPLIT2_STRONG_FACTOR = 2
SPLIT2_FIRST_GAP_MULT = 3
SPLIT2_FIRST_MIN_RATIO = 0.5
SPLIT_CLIP_SLACK = 12
SPLIT_CORE_LEASH = 16
TWIN_GUTTER_MIN = 4
TWIN_SIDE_MIN = 2
TWIN_SPAN_MIN = 48


def _r(x):
    return math.floor(x + 0.5)


def _med(xs):
    s = sorted(xs)
    return s[len(s) // 2]


def _bbox(rs):
    return {"x1": min(r["x1"] for r in rs), "y1": min(r["y1"] for r in rs),
            "x2": max(r["x2"] for r in rs), "y2": max(r["y2"] for r in rs)}


def split_merged_boxes(boxes, comps, same_block_gap, box_comps=None):
    if box_comps is None:
        box_comps = comps
    if len(comps) < 2:
        return list(boxes)

    def centered(c, b):
        return ((c["x1"] + c["x2"]) / 2 >= b["x1"]
                and (c["x1"] + c["x2"]) / 2 <= b["x2"]
                and (c["y1"] + c["y2"]) / 2 >= b["y1"]
                and (c["y1"] + c["y2"]) / 2 <= b["y2"])

    out = []
    for b in boxes:
        cs = [c for c in comps if centered(c, b)]
        bs = cs if box_comps is comps else [c for c in box_comps if centered(c, b)]
        parts = _split_box(b, cs, same_block_gap, bs)
        out.extend(parts if parts is not None else [b])
    return out


def _emit_split(box, groups, axis, loose, box_comps):
    lo = (lambda g: g["y1"]) if axis == "y" else (lambda g: g["x1"])
    hi = (lambda g: g["y2"]) if axis == "y" else (lambda g: g["x2"])

    def gap_before(i):
        if i <= 0 or i >= len(groups):
            return float("inf")
        return lo(groups[i]) - hi(groups[i - 1])

    cuts = []
    for i, g in enumerate(groups[1:]):
        gap = lo(g) - hi(groups[i])
        cuts.append({"at": (hi(groups[i]) + lo(g)) / 2,
                     "slack": min(SPLIT_CLIP_SLACK, max(4, math.floor(gap / 2)))})
    kids = []
    for i, g in enumerate(groups):
        def pad_to(gap):
            return min(SPLIT_PAD_CAP, math.floor(gap / 2)) \
                if gap > 0 and math.isfinite(gap) else 0

        pad_before = pad_to(gap_before(i))
        pad_after = pad_to(gap_before(i + 1))

        def in_group(c):
            return ((c["x1"] + c["x2"]) / 2 >= g["x1"]
                    and (c["x1"] + c["x2"]) / 2 <= g["x2"]
                    and (c["y1"] + c["y2"]) / 2 >= g["y1"]
                    and (c["y1"] + c["y2"]) / 2 <= g["y2"])

        own = [c for c in box_comps if in_group(c)]
        ext = dict(g)
        if own:
            core = _bbox(own)
            near = [c for c in loose if in_group(c)
                    and c["x1"] <= core["x2"] + SPLIT_CORE_LEASH
                    and c["x2"] >= core["x1"] - SPLIT_CORE_LEASH
                    and c["y1"] <= core["y2"] + SPLIT_CORE_LEASH
                    and c["y2"] >= core["y1"] - SPLIT_CORE_LEASH]
            if near:
                ext = _bbox(near)
        clip = {"x1": box["x1"], "y1": box["y1"], "x2": box["x2"], "y2": box["y2"]}
        before = cuts[i - 1] if i > 0 else None
        after = cuts[i] if i < len(groups) - 1 else None
        if axis == "y":
            if before:
                clip["y1"] = _r(before["at"] - before["slack"])
            if after:
                clip["y2"] = _r(after["at"] + after["slack"])
        else:
            if before:
                clip["x1"] = _r(before["at"] - before["slack"])
            if after:
                clip["x2"] = _r(after["at"] + after["slack"])
        if axis == "y":
            r = {"x1": max(box["x1"], ext["x1"]),
                 "y1": max(box["y1"], ext["y1"] - pad_before),
                 "x2": min(box["x2"], ext["x2"]),
                 "y2": min(box["y2"], ext["y2"] + pad_after)}
        else:
            r = {"x1": max(box["x1"], ext["x1"] - pad_before),
                 "y1": max(box["y1"], ext["y1"]),
                 "x2": min(box["x2"], ext["x2"] + pad_after),
                 "y2": min(box["y2"], ext["y2"])}
        if axis == "y":
            if before:
                r["y1"] = max(r["y1"], _r(before["at"]))
            if after:
                r["y2"] = min(r["y2"], _r(after["at"]))
        else:
            if before:
                r["x1"] = max(r["x1"], _r(before["at"]))
            if after:
                r["x2"] = min(r["x2"], _r(after["at"]))
        kid = dict(box)
        kid.update(r)
        kid["clip"] = clip
        kid["cutAxis"] = axis
        kids.append(kid)
    return kids


def _split_box(box, cs, same_block_gap, box_comps):
    if len(cs) < 2:
        return None
    return (_split_box_lane1(box, cs, same_block_gap, box_comps)
            or _split_box_lane2(box, cs, box_comps)
            or _split_twin_cut(box, cs, box_comps))


def _split_twin_cut(box, cs, box_comps):
    cl = [dict(c, x1=max(c["x1"], box["x1"]), x2=min(c["x2"], box["x2"])) for c in cs]
    edges = []
    for c in cl:
        if c["x2"] <= c["x1"]:
            continue
        edges.append((c["x1"], True))
        edges.append((c["x2"], False))
    # closes sort before opens at ties, so touching comps leave no avenue
    edges.sort(key=lambda e: (e[0], 1 if e[1] else 0))
    ivs = []
    depth, start = 0, box["x1"]
    for x, is_open in edges:
        if x < box["x1"] or x > box["x2"]:
            continue
        if is_open:
            if depth == 0 and x - start >= TWIN_GUTTER_MIN:
                ivs.append((start, x))
            depth += 1
        else:
            depth -= 1
            if depth == 0:
                start = x
    if depth == 0 and box["x2"] - start >= TWIN_GUTTER_MIN:
        ivs.append((start, box["x2"]))
    for a, b in ivs:
        mid = (a + b) / 2
        L = [c for c in cl if c["x2"] <= mid and c["x2"] - c["x1"] > c["y2"] - c["y1"]]
        R = [c for c in cl if c["x1"] >= mid and c["x2"] - c["x1"] > c["y2"] - c["y1"]]
        if len(L) < TWIN_SIDE_MIN or len(R) < TWIN_SIDE_MIN:
            continue
        span = (lambda ss: max(c["y2"] for c in ss) - min(c["y1"] for c in ss))
        if span(L) < TWIN_SPAN_MIN or span(R) < TWIN_SPAN_MIN:
            continue
        return _emit_split(box, [_bbox(L), _bbox(R)], "x", cs, box_comps)
    return None


def _split_box_lane1(box, cs, same_block_gap, box_comps):
    for axis in ("y", "x"):
        lo = (lambda c: c["y1"]) if axis == "y" else (lambda c: c["x1"])
        hi = (lambda c: c["y2"]) if axis == "y" else (lambda c: c["x2"])
        c_lo = (lambda c: c["x1"]) if axis == "y" else (lambda c: c["y1"])
        c_hi = (lambda c: c["x2"]) if axis == "y" else (lambda c: c["y2"])
        srt = sorted(cs, key=lo)
        thr = max(SPLIT_GAP_FACTOR * same_block_gap,
                  SPLIT_GAP_RATIO * _med([hi(c) - lo(c) for c in srt]))
        groups = []
        for c in srt:
            g = groups[-1] if groups else None
            gap = lo(c) - (g["y2"] if axis == "y" else g["x2"]) if g else 0
            if g and gap >= thr:
                groups.append(dict(c))
            elif g:
                g["x1"] = min(g["x1"], c["x1"])
                g["y1"] = min(g["y1"], c["y1"])
                g["x2"] = max(g["x2"], c["x2"])
                g["y2"] = max(g["y2"], c["y2"])
            else:
                groups.append(dict(c))
        if len(groups) < 2:
            continue
        while True:
            k = next((k for k in range(len(groups) - 1)
                      if min(c_hi(groups[k]), c_hi(groups[k + 1]))
                      > max(c_lo(groups[k]), c_lo(groups[k + 1]))), -1)
            if k < 0:
                break
            b = groups.pop(k + 1)
            groups[k] = {"x1": min(groups[k]["x1"], b["x1"]),
                         "y1": min(groups[k]["y1"], b["y1"]),
                         "x2": max(groups[k]["x2"], b["x2"]),
                         "y2": max(groups[k]["y2"], b["y2"])}
        if len(groups) < 2:
            continue
        return _emit_split(box, groups, axis, cs, box_comps)
    return None


def _split_box_lane2(box, cs, box_comps):
    if len(cs) < 2:
        return None
    unit = _med([min(c["x2"] - c["x1"], c["y2"] - c["y1"]) for c in cs])
    floor = max(SPLIT2_FLOOR_MIN, _r(SPLIT2_FLOOR_RATIO * unit))
    for axis in ("y", "x"):
        lo = (lambda g: g["y1"]) if axis == "y" else (lambda g: g["x1"])
        hi = (lambda g: g["y2"]) if axis == "y" else (lambda g: g["x2"])
        c_lo = (lambda g: g["x1"]) if axis == "y" else (lambda g: g["y1"])
        c_hi = (lambda g: g["x2"]) if axis == "y" else (lambda g: g["y2"])
        srt = sorted(cs, key=lo)
        groups = []
        for c in srt:
            g = groups[-1] if groups else None
            gap = lo(c) - hi(g) if g else 0
            if g and gap >= floor:
                groups.append(dict(c))
            elif g:
                g["x1"] = min(g["x1"], c["x1"])
                g["y1"] = min(g["y1"], c["y1"])
                g["x2"] = max(g["x2"], c["x2"])
                g["y2"] = max(g["y2"], c["y2"])
            else:
                groups.append(dict(c))
        if len(groups) < 2:
            continue
        merged = [groups[0]]
        for i in range(1, len(groups)):
            prev, g = merged[-1], groups[i]
            gap = lo(g) - hi(prev)
            ov = min(c_hi(prev), c_hi(g)) - max(c_lo(prev), c_lo(g))
            span = min(c_hi(prev) - c_lo(prev), c_hi(g) - c_lo(g))
            ratio = 0 if ov <= 0 else ov / span
            nested = ((c_lo(prev) >= c_lo(g) and c_hi(prev) <= c_hi(g))
                      or (c_lo(g) >= c_lo(prev) and c_hi(g) <= c_hi(prev)))
            first_pair = (axis == "y" and len(merged) == 1 and i == 1
                            and nested and gap >= SPLIT2_FIRST_GAP_MULT * floor
                            and hi(g) - lo(g) >= (hi(prev) - lo(prev)) * SPLIT2_FIRST_MIN_RATIO)
            if gap >= floor and (ratio < SPLIT2_OVERLAP_MAX
                                 or (gap >= SPLIT2_STRONG_FACTOR * floor and not nested)
                                 or first_pair):
                merged.append(g)
            else:
                prev["x1"] = min(prev["x1"], g["x1"])
                prev["y1"] = min(prev["y1"], g["y1"])
                prev["x2"] = max(prev["x2"], g["x2"])
                prev["y2"] = max(prev["y2"], g["y2"])
        if len(merged) >= 2:
            return _emit_split(box, merged, axis, cs, box_comps)
    return None

# ---- OCR-crop expansion (ported 1:1 from expandCropToInk in
# src/content/render.ts) ----
# A detection box can clip its own glyphs, so after padding any side whose
# edge still touches ink grows outward to the last ink plus a small margin,
# capped at half the box's smaller side. Split children stay inside their
# clip. Only the READ window grows: crops stay tight when nothing is cut.
# rgb is a full-page uint8 HxWx3 array; box holds the detection bounds (+clip
# on split children); rect is the padded crop {x,y,w,h}.
def _crop_expand_cap(box):
    return max(16, _r(min(box["x2"] - box["x1"], box["y2"] - box["y1"]) * 0.5))


def _interior_seed(rgb, box):
    H, W, _ = rgb.shape
    x1, y1 = max(0, math.floor(box["x1"])), max(0, math.floor(box["y1"]))
    x2, y2 = min(W - 1, math.ceil(box["x2"])), min(H - 1, math.ceil(box["y2"]))
    step_x = max(1, (x2 - x1) // 24)
    step_y = max(1, (y2 - y1) // 24)
    buckets = {}
    best = None
    for y in range(y1, y2 + 1, step_y):
        for x in range(x1, x2 + 1, step_x):
            r, g, b = (int(v) for v in rgb[y, x])
            key = ((r >> 4) << 8) | ((g >> 4) << 4) | (b >> 4)
            bkt = buckets.get(key)
            if bkt is None:
                bkt = [0, 0, 0, 0]
                buckets[key] = bkt
            bkt[0] += 1
            bkt[1] += r
            bkt[2] += g
            bkt[3] += b
            if best is None or bkt[0] > best[0]:
                best = bkt
    if best is None:
        cx = max(0, min(W - 1, math.floor((box["x1"] + box["x2"]) / 2)))
        cy = max(0, min(H - 1, math.floor((box["y1"] + box["y2"]) / 2)))
        return [float(v) for v in rgb[cy, cx]]
    return [best[1] / best[0], best[2] / best[0], best[3] / best[0]]


def expand_crop_to_ink(rgb, box, rect):
    H, W, _ = rgb.shape
    seed = _interior_seed(rgb, box)
    seed_lum = 0.299 * seed[0] + 0.587 * seed[1] + 0.114 * seed[2]
    cap = _crop_expand_cap(box)
    clip = box.get("clip")
    x1 = max(0, math.floor(rect["x"]))
    y1 = max(0, math.floor(rect["y"]))
    x2 = min(W - 1, math.ceil(rect["x"] + rect["w"]))
    y2 = min(H - 1, math.ceil(rect["y"] + rect["h"]))

    def ink_at(x, y):
        if x < 0 or y < 0 or x >= W or y >= H:
            return False
        r, g, b = (float(v) for v in rgb[y, x])
        return abs(0.299 * r + 0.587 * g + 0.114 * b - seed_lum) >= 90

    def edge_ink(vertical, at, lo, hi):
        return [q for q in range(lo, hi + 1) if (ink_at(at, q) if vertical else ink_at(q, at))]

    def connected_mass(vertical, pts, at):
        lo_x, hi_x = max(0, x1 - cap), min(W - 1, x2 + cap)
        lo_y, hi_y = max(0, y1 - cap), min(H - 1, y2 + cap)
        ix1, iy1 = math.ceil(box["x1"]) + 2, math.ceil(box["y1"]) + 2
        ix2, iy2 = math.floor(box["x2"]) - 2, math.floor(box["y2"]) - 2
        seen = bytearray(W * H)
        dist = {}
        queue = []
        for q in pts:
            x, y = (at, q) if vertical else (q, at)
            if x < lo_x or x > hi_x or y < lo_y or y > hi_y or seen[y * W + x]:
                continue
            seen[y * W + x] = 1
            dist[x + y * W] = 0
            queue.append(x + y * W)
        reached = False
        bx1, by1, bx2, by2 = W, H, -1, -1
        head = 0
        while head < len(queue):
            p = queue[head]
            head += 1
            x, y = p % W, p // W
            if not ink_at(x, y):
                continue
            d = dist[p]
            bx1, by1 = min(bx1, x), min(by1, y)
            bx2, by2 = max(bx2, x), max(by2, y)
            if ix1 <= x <= ix2 and iy1 <= y <= iy2:
                reached = True
            if d + 1 > cap:
                continue
            for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
                if nx < lo_x or nx > hi_x or ny < lo_y or ny > hi_y:
                    continue
                np_ = nx + ny * W
                if not seen[np_]:
                    seen[np_] = 1
                    dist[np_] = d + 1
                    queue.append(np_)
        if reached and bx2 >= bx1:
            return {"x1": bx1, "y1": by1, "x2": bx2, "y2": by2}
        return None

    bounds = {"x1": x1, "y1": y1, "x2": x2, "y2": y2}

    def grow(side):
        vertical = side in ("l", "r")
        d = -1 if side in ("l", "t") else 1
        edge = bounds["x1"] if side == "l" else bounds["x2"] if side == "r" \
            else bounds["y1"] if side == "t" else bounds["y2"]
        lo, hi = (bounds["y1"], bounds["y2"]) if vertical else (bounds["x1"], bounds["x2"])
        if side == "l":
            bound = max(0, math.ceil(box["x1"]) - cap, math.ceil(clip["x1"]) if clip else 0)
        elif side == "r":
            bound = min(W - 1, math.floor(box["x2"]) + cap, math.floor(clip["x2"]) if clip else W - 1)
        elif side == "t":
            bound = max(0, math.ceil(box["y1"]) - cap, math.ceil(clip["y1"]) if clip else 0)
        else:
            bound = min(H - 1, math.floor(box["y2"]) + cap, math.floor(clip["y2"]) if clip else H - 1)
        touch = edge_ink(vertical, edge, lo, hi)
        if not touch or len(touch) >= (hi - lo + 1) * 0.6:
            return
        mass = connected_mass(vertical, touch, edge)
        if not mass:
            return
        grown = mass["x1"] - 4 if side == "l" else mass["x2"] + 4 if side == "r" \
            else mass["y1"] - 4 if side == "t" else mass["y2"] + 4
        limited = max(grown, edge - cap) if d < 0 else min(grown, edge + cap)
        if side == "l":
            bounds["x1"] = min(edge, max(limited, bound))
        elif side == "r":
            bounds["x2"] = max(edge, min(limited, bound))
        elif side == "t":
            bounds["y1"] = min(edge, max(limited, bound))
        else:
            bounds["y2"] = max(edge, min(limited, bound))

    for side in ("l", "r", "t", "b"):
        grow(side)
    return {"x": bounds["x1"], "y": bounds["y1"],
            "w": bounds["x2"] - bounds["x1"], "h": bounds["y2"] - bounds["y1"]}


# ---- mask packing (mirrors packMask in src/content/page-cache.ts) ----
# Block-max downscale of the binary text mask to <=256 on the long side;
# the client restores it with unpackMask. /v1/page ships this so cloud
# entries carry a REAL mask (text-color sampling, inpaint and the debug view
# all read it) instead of the client's old box-filled stand-in, which
# excluded every whole box from sampling and forced white text. Standard
# library only: flat is any row-major byte sequence (bytes, bytearray, or a
# ravelled numpy array).
def pack_mask(w, h, flat, max_side=256):
    step = max(1, max(w, h) // max_side)
    ow, oh = (w + step - 1) // step, (h + step - 1) // step
    out = bytearray(ow * oh)
    for y in range(oh):
        y0, y1 = y * step, min(y * step + step, h)
        for x in range(ow):
            x0, x1 = x * step, min(x * step + step, w)
            v = 0
            for yy in range(y0, y1):
                base = yy * w
                for xx in range(x0, x1):
                    if flat[base + xx] > v:
                        v = flat[base + xx]
                        if v > 127:
                            break
                if v > 127:
                    break
            out[y * ow + x] = 255 if v > 127 else 0
    return ow, oh, bytes(out)


# ---- pass-3 rescue (mirrors rescueSplitComp in src/content/detection.ts) ----
# A merged comp killed ONLY by the overlap gate may still hold a text group
# outside every kept box: split it with the lane machinery on the raw texty
# comps and re-gate each piece. count_in recounts the parent comp's labels
# inside a piece bbox; overlaps_box and box_conf are the caller's gates.
def rescue_split_comp(c, texty, strict, same_block_gap, page_area,
                      count_in, overlaps_box, box_conf):
    pieces = split_merged_boxes(
        [dict(c, conf=0.5)], texty, same_block_gap, strict)
    if len(pieces) < 2:
        return []
    out = []
    for pc in pieces:
        bw, bh = pc["x2"] - pc["x1"], pc["y2"] - pc["y1"]
        if bw < 14 or bh < 14 or bw * bh > 0.2 * page_area:
            continue
        count, psum = count_in(math.floor(pc["x1"]), math.floor(pc["y1"]),
                               math.ceil(pc["x2"]), math.ceil(pc["y2"]))
        if count / (bw * bh) < 0.02:
            continue
        m = {"x1": pc["x1"], "y1": pc["y1"], "x2": pc["x2"], "y2": pc["y2"]}
        if overlaps_box(m):
            continue
        if psum / count < 0.75 and box_conf(m) < 0.20:
            continue
        out.append({"x1": pc["x1"], "y1": pc["y1"], "x2": pc["x2"],
                    "y2": pc["y2"], "conf": 0.5})
    return out


In [ ]:
%%writefile serve.py
# Standalone launcher for platforms with no wrapper of their own (Colab's own
# GPU — server/colab-server.ipynb — or any VPS). Modal keeps modal_app.py.
# Same Bearer rule either way: ARN_API_KEY set = auth on, / and /health stay
# open (the extension reads /health for Test/warm), everything else 401s.
import os

from app import app

KEY = os.environ.get("ARN_API_KEY", "")

if KEY:
    from starlette.middleware.base import BaseHTTPMiddleware
    from starlette.responses import JSONResponse

    class Auth(BaseHTTPMiddleware):
        async def dispatch(self, request, call_next):
            if request.url.path in ("/", "/health"):
                return await call_next(request)
            if request.headers.get("authorization") != f"Bearer {KEY}":
                return JSONResponse({"ok": False, "error": "unauthorized"}, 401)
            return await call_next(request)

    app.add_middleware(Auth)
else:
    print("serve: ARN_API_KEY not set — running WITHOUT auth (local/testing only)",
          flush=True)

if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", "7860")))


## 3. Download the models (~230MB)

Fetched fresh every session (the VM is wiped between sessions) and
skipped when the files are already here.


In [ ]:
FILES = [
    ("ctd.onnx", "https://huggingface.co/lemondouble/lemon-manga-translator/resolve/main/onnx/comic-text-detector/ctd.onnx?download=true"),
    ("lama-manga-512-fp16w.onnx", "https://huggingface.co/c0ffeeOverdose/arn-manga-models/resolve/main/lama-manga-512-fp16w.onnx?download=true"),
    ("vision-int4.onnx", "https://huggingface.co/genshiai-daichi/baberu-ocr/resolve/main/onnx/vision_int4.onnx?download=true"),
    ("baberu-prefill.onnx", "https://huggingface.co/genshiai-daichi/baberu-ocr/resolve/main/onnx/decoder_prefill_int8.onnx?download=true"),
    ("baberu-step.onnx", "https://huggingface.co/genshiai-daichi/baberu-ocr/resolve/main/onnx/decoder_step_int8.onnx?download=true"),
    ("vocab.json", "https://huggingface.co/genshiai-daichi/baberu-ocr/resolve/main/tokenizer/vocab.json?download=true"),
]
import os, subprocess
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
for name, url in FILES:
    dest = os.path.join(MODEL_DIR, name)
    if os.path.isfile(dest) and os.path.getsize(dest) > 0:
        print("have", name)
        continue
    part = dest + ".part"
    r = subprocess.run(["curl", "-fL", "--retry", "3", "--retry-all-errors", "-o", part, url])
    assert r.returncode == 0, f"download failed: {name}"
    os.replace(part, dest)
    print("got", name, str(os.path.getsize(dest) // 1048576) + "MB")
print("models ready in", MODEL_DIR)


## 4. Start the server

Loads the models onto the GPU (tens of seconds on a T4) and keeps the
server running in the background. Re-running this cell restarts the
server, so the files saved in step 2 always take effect (an already-up
server would otherwise keep running older code). The API key is saved
to `/content/api_key.txt` so re-runs keep the same one.


In [ ]:
import glob, json, os, secrets, signal, subprocess, sys, time, urllib.request

PORT = 7860
keyfile = "/content/api_key.txt"
pidfile = "/content/server.pid"
if os.path.isfile(keyfile):
    API_KEY = open(keyfile).read().strip()
else:
    API_KEY = secrets.token_hex(32)
    open(keyfile, "w").write(API_KEY)

# Colab ships CUDA/cuDNN as pip packages under site-packages/nvidia —
# the loader needs those dirs on its path for ORT's CUDA provider
cuda_libs = sorted(set(
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
    + glob.glob("/usr/local/lib/python*/site-packages/nvidia/*/lib")))

def health():
    try:
        with urllib.request.urlopen("http://127.0.0.1:" + str(PORT) + "/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

# a previous run's server keeps running across cell re-runs — kill it
# so the files from step 2 (not older code) are what comes up
if os.path.isfile(pidfile):
    try:
        old = int(open(pidfile).read().strip())
    except Exception:
        old = None
    if old and pid_alive(old):
        print("stopping the previous server (pid %d)..." % old)
        try:
            os.kill(old, signal.SIGTERM)
        except Exception:
            pass
        for _ in range(30):
            if not pid_alive(old):
                break
            time.sleep(1)
    try:
        os.remove(pidfile)
    except Exception:
        pass

if health() is None:
    env = dict(os.environ, MODEL_DIR="/content/models", ARN_API_KEY=API_KEY, PORT=str(PORT),
               ORT_PROVIDERS="CUDAExecutionProvider,CPUExecutionProvider", ORT_DEVICE="cuda",
               LD_LIBRARY_PATH=":".join(cuda_libs + [os.environ.get("LD_LIBRARY_PATH", "")]))
    log = open("/content/server.log", "w")
    proc = subprocess.Popen([sys.executable, "serve.py"], env=env,
                            stdout=log, stderr=subprocess.STDOUT)
    open(pidfile, "w").write(str(proc.pid))
    t0 = time.time()
    while time.time() - t0 < 240 and health() is None:
        if proc.poll() is not None:
            print(open("/content/server.log").read()[-3000:])
            raise SystemExit("server exited — see log above")
        time.sleep(2)
    assert health() is not None, "server did not come up in 240s — see /content/server.log"
h = health()
eps = h.get("ep") or {}
print("✓ server up — ep=%s splitGen=%s" % (eps, h.get("splitGen")))
if not any("CUDAExecutionProvider" in v for v in eps.values()):
    print("! CUDA is not active (CPU only) — pages will be very slow. Re-run the install cell, or Runtime → Change runtime type → T4 GPU.")


## 5. Open the tunnel

Cloudflare quick tunnel — no account needed. The URL is new every time
you run this cell; the old ones keep working until the session ends.


In [ ]:
import os, re, subprocess, time
BIN = "/content/cloudflared"
if not os.path.isfile(BIN):
    r = subprocess.run(["curl", "-fL", "--retry", "3", "-o", BIN,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    assert r.returncode == 0, "cloudflared download failed"
    os.chmod(BIN, 0o755)

LOG = "/content/tunnel.log"
with open(LOG, "w") as log:
    proc = subprocess.Popen([BIN, "tunnel", "--url", "http://127.0.0.1:" + str(PORT), "--no-autoupdate"],
                            stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while time.time() - t0 < 45:
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(LOG).read())
    if m:
        ENDPOINT = m.group(0)
        break
    if proc.poll() is not None:
        print(open(LOG).read()[-3000:])
        raise SystemExit("cloudflared exited — see log above")
assert "ENDPOINT" in dir(), "no tunnel URL in 45s — see /content/tunnel.log"
print("✓ tunnel:", ENDPOINT)


## 6. Test through the tunnel

Sends one tiny image through the public URL and prints the two values
for the extension.


In [ ]:
import io, json, time, urllib.error, urllib.request
from PIL import Image

img = Image.new("L", (64, 64), 128)
buf = io.BytesIO()
img.save(buf, "JPEG")
blob = buf.getvalue()

def post(headers, timeout):
    req = urllib.request.Request(ENDPOINT + "/v1/page", data=blob, headers=headers)
    return urllib.request.urlopen(req, timeout=timeout)

# a fresh quick-tunnel hostname can lag in the VM's resolver — retry
# before calling it broken (the URL is public, so the API key is the
# only gate: verify it is enforced on the way)
res = None
for attempt in range(8):
    try:
        try:
            post({}, 30)
            raise SystemExit("auth is NOT enforced — check ARN_API_KEY / serve.py")
        except urllib.error.HTTPError as e:
            assert e.code == 401, "expected 401 without a key, got %s" % e.code
        with post({"Authorization": "Bearer " + API_KEY,
                   "Content-Type": "image/jpeg"}, 180) as r:
            res = json.load(r)
        break
    except urllib.error.URLError as e:
        print("tunnel not resolvable yet (attempt %d): %s" % (attempt + 1, e))
        time.sleep(6)

if res is None:
    print("! could not reach the endpoint from inside this VM — external clients")
    print("  usually still work; try the extension's Test button first.")
else:
    assert res.get("ok") and "boxes" in res, res
    print("✓ endpoint answers through the tunnel: %dx%d, %d boxes, %.0fms total"
          % (res["w"], res["h"], len(res["boxes"]), res["ms"]["total"]))
print("=" * 60)
print("ENDPOINT:", ENDPOINT)
print("API KEY :", API_KEY)
print("=" * 60)
print("Paste both into the extension: Options → Model → Where detection runs → Cloud,")
print("then press Test cloud & prewarm. Keep this tab open while you read.")


## If something goes wrong

- **Server log / crash:** `open('/content/server.log').read()[-2000:]`
  in a new cell.
- **Tunnel log:** same with `/content/tunnel.log`.
- **Colab disconnected / VM died:** Runtime → Reconnect (or Run all
  again) — the models re-download, the tunnel prints a NEW url; paste
  it into the extension.
- **The extension says the request failed:** re-run from step 4 down,
  check the endpoint URL matches the last banner, and press Test cloud
  & prewarm in the extension options.
- **Very slow pages:** `ep` at step 4 showed only
  `CPUExecutionProvider` — the pinned `onnxruntime-gpu` wheel did not
  match the runtime's CUDA. Re-run the install cell, then re-run step
  4 down.
- **Want a stable URL instead?** Deploy with `cloud-setup.ipynb`
  (Modal).
